In [1]:
# %%
# ============================================================
# 0. OpenAI API setup
# ============================================================

from dotenv import load_dotenv
from openai import OpenAI
import os


load_dotenv(
    "/Users/yurujia/Desktop/Dissertation Data/sentiment/.env"
)


key = os.getenv(
    "OPENAI_API_KEY"
)


if key:
    print("API key loaded successfully")
else:
    print("API key NOT found")


client = OpenAI()


MODEL = "gpt-5-mini"

print(
    "Model:",
    MODEL
)

API key loaded successfully
Model: gpt-5-mini


In [2]:
# %%
# ============================================================
# 1. Imports
# ============================================================

import pandas as pd
from pathlib import Path
import json


print("Packages loaded.")

Packages loaded.


In [3]:
# %%
# ============================================================
# 2. Paths
# ============================================================


BASE_DIR = Path(
    "/Users/yurujia/Desktop/Dissertation Data/China"
)


INPUT_PATH = (
    BASE_DIR /
    "descriptive_stats/xinhua_av_final_with_first_av_relevant_paragraph.xlsx"
)


BATCH_INPUT_PATH = (
    BASE_DIR /
    "xinhua_sentiment_P2_CN_batch_input.jsonl"
)


BATCH_OUTPUT_PATH = (
    BASE_DIR /
    "xinhua_sentiment_P2_CN_batch_output.jsonl"
)


OUTPUT_PATH = (
    BASE_DIR /
    "xinhua_sentiment_P2_CN_final.xlsx"
)


print(INPUT_PATH)

/Users/yurujia/Desktop/Dissertation Data/China/descriptive_stats/xinhua_av_final_with_first_av_relevant_paragraph.xlsx


In [4]:
# %%
# ============================================================
# 3. Load dataset
# ============================================================


df = pd.read_excel(
    INPUT_PATH
)


print(
    "Dataset shape:",
    df.shape
)


print(
    df.columns.tolist()
)


display(
    df.head()
)

Dataset shape: (1155, 88)
['file', 'title', 'source', 'date', 'content', 'source_file', 'has_body_marker', 'has_wan_marker', 'has_load_date_marker', 'article_content_clean', 'extraction_method', 'extract_success', 'date_original', 'date_fixed', 'date_parsed_direct', 'year_temp', 'month_temp', 'content_char_length', 'approx_token_count', 'exclude_title_keyword', 'exclude_too_long', 'exclude_from_analysis', 'exclusion_reason', 'has_explicit_brief_title', 'has_explicit_brief_header', 'small_heading_count', 'has_multiple_small_headings', 'exclude_multi_topic_brief', 'multi_topic_brief_exclusion_reason', 'automated_screen_text', 'has_general_road_av_term', 'has_higher_automation_term', 'has_passenger_robotaxi_term', 'has_strong_passenger_robotaxi_term', 'has_bus_public_transport_term', 'has_adas_term', 'has_road_vehicle_context', 'has_strong_non_road_term', 'has_general_non_road_context', 'has_non_road_primary_topic_term', 'has_non_road_term', 'remove_no_relevant_road_av_term', 'remove_pure

,file,title,source,date,content,source_file,has_body_marker,has_wan_marker,has_load_date_marker,article_content_clean,...,av_paragraphs_checked_before_match,av_nonrelevant_substantive_paragraphs_skipped,av_skipped_nonrelevant_text,first_av_relevant_paragraph_char_count,first_av_relevant_paragraph_chinese_char_count,first_av_relevant_paragraph_english_word_count,first_av_relevant_paragraph_number_count,first_av_relevant_paragraph_approx_text_unit_count,first_paragraph_is_av_relevant,av_paragraph_same_as_original_first
0,（财经）专访：“中德电动汽车合作远大于竞争”——访蔚来汽车董事长李斌.DOCX,（财经）专访：“中德电动汽车合作远大于竞争”——访蔚来汽车董事长李斌,Xinhua: News in Chinese for Overseas Service,"June 2, 2017 Friday 3:43 AM GMT",（财经）专访：“中德电动汽车合作远大于竞争”——访蔚来汽车董事长李斌\nXinhua: Ne...,lexis_structured1-200,True,True,True,新华社柏林６月１日电专访：“中德电动汽车合作远大于竞争”——访蔚来汽车董事长李斌\n新华社记...,...,2,1,中国和德国在电动汽车领域各有优势，两国车企如果能够融合优势，将创造领先全球的强大竞争力，未来...,128,114,0,3,117,False,False
1,（李克强出访配合稿）专访：务实合作始终是中德关系的基本特征——访中国驻德国公使衔参赞王卫东....,（李克强出访配合稿）专访：务实合作始终是中德关系的基本特征——访中国驻德国公使衔参赞王卫东,Xinhua: News in Chinese for Overseas Service,"June 3, 2017 Saturday 4:15 AM GMT",（李克强出访配合稿）专访：务实合作始终是中德关系的基本特征——访中国驻德国公使衔参赞王卫东\...,lexis_structured1-200,True,True,True,新华社柏林６月２日电专访：务实合作始终是中德关系的基本特征——访中国驻德国公使衔参赞王卫东\...,...,5,4,“经贸合作始终是中德关系的压舱石。李克强总理此访在经贸领域取得了丰硕的成果，必将推动中德经贸...,139,122,0,0,122,False,False
2,中国新能源汽车全球占比一半.DOCX,中国新能源汽车全球占比一半,Xinhua: News in Chinese for Overseas Service,"June 6, 2017 Tuesday 8:29 AM GMT",中国新能源汽车全球占比一半\nXinhua: News in Chinese for Ove...,lexis_structured1-200,True,True,True,新华社北京６月６日电（记者陈芳 董瑞丰）通过手机定位，找到距离最近的电动汽车，用手机开锁后就...,...,6,5,通过手机定位，找到距离最近的电动汽车，用手机开锁后就可以驾驶，直至停下来留给下一位客户使用。...,43,38,0,0,38,False,False
3,（科技）专家：自动驾驶技术有望让交通事故零伤亡.DOCX,（科技）专家：自动驾驶技术有望让交通事故零伤亡,Xinhua: News in Chinese for Overseas Service,"June 7, 2017 Wednesday 7:24 AM GMT",（科技）专家：自动驾驶技术有望让交通事故零伤亡\nXinhua: News in Chine...,lexis_structured1-200,True,True,True,新华社瑞典哥德堡６月７日电（记者潘革平 付一鸣）吉利欧洲研发中心首席执行官方浩瀚日前在位于瑞...,...,1,0,NaN,79,72,0,1,73,True,True
4,（财经）日本预计于２０２３年普及５Ｇ通信.DOCX,（财经）日本预计于２０２３年普及５Ｇ通信,Xinhua: News in Chinese for Overseas Service,"June 7, 2017 Wednesday 6:42 AM GMT",（财经）日本预计于２０２３年普及５Ｇ通信\nXinhua: News in Chinese ...,lexis_structured1-200,True,True,True,新华社东京６月７日电（记者钱铮）下一代超高速无线通信技术——第五代移动通信技术（５Ｇ）的商业...,...,1,0,NaN,72,60,0,2,62,True,True


In [5]:
# %%
# ============================================================
# 4. Prepare sentiment dataframe
# ============================================================


TEXT_COL = (
    "first_av_relevant_paragraph"
)


assert TEXT_COL in df.columns



sentiment_df = pd.DataFrame({

    "article_id":
        range(len(df)),


    "source":
        "Xinhua",


    "text":
        df[TEXT_COL]

})


sentiment_df = (
    sentiment_df
    .dropna(
        subset=["text"]
    )
)


sentiment_df["text"] = (
    sentiment_df["text"]
    .astype(str)
    .str.strip()
)


sentiment_df = sentiment_df[
    sentiment_df["text"].str.len() > 0
]


sentiment_df = (
    sentiment_df
    .reset_index(drop=True)
)


print(
    "Articles for classification:",
    len(sentiment_df)
)


display(
    sentiment_df.head()
)

Articles for classification: 1155


,article_id,source,text
0,0,Xinhua,在李克强总理访问德国期间，蔚来汽车５月３１日在柏林与德国大陆集团签署战略合作协议，主要涉及电...
1,1,Xinhua,王卫东说，中德双方应“共塑创新”，构建双边关系发展新引擎。双方在智能制造、人工智能、自动驾驶...
2,2,Xinhua,“我们正在逐渐把自动驾驶融入电动汽车这个重大专项、融入未来电动汽车产品当中。”万钢说。
3,3,Xinhua,吉利欧洲研发中心首席执行官方浩瀚日前在位于瑞典哥德堡的总部对新华社记者表示，未来汽车应用自动...
4,4,Xinhua,下一代超高速无线通信技术——第五代移动通信技术（５Ｇ）的商业使用区域预计将于２０２３年扩大至...


In [6]:
# %%
# ============================================================
# 5. Final P2-CN prompt
# ============================================================


def build_prompt_p2_cn(text):

    return f"""
你正在为一项关于自动驾驶新闻报道的学术研究进行情感分类。

你的任务是判断以下新闻文本对自动驾驶汽车、自动驾驶技术及其发展与应用所表达的情感倾向。

请使用以下定义：

1 = 正面

文本整体以较为积极、有利的方式呈现自动驾驶汽车或自动驾驶技术。

这可能包括：
- 技术进步；
- 创新发展；
- 社会或经济效益；
- 安全性提升；
- 成功测试或部署；
- 商业扩张；
- 支持性发展；
- 对未来发展的积极预期。


0 = 中性

文本整体以事实性、描述性、平衡性或混合性的方式报道自动驾驶，
没有明确表现出占主导地位的正面或负面评价。

包括：
- 事实报道；
- 技术介绍；
- 企业公告；
- 政策信息；
- 项目进展；
- 测试或合作信息；

但没有明显评价方向。


-1 = 负面

文本整体以较为消极、不利的方式呈现自动驾驶汽车或自动驾驶技术。

包括：
- 技术失败；
- 安全风险；
- 自动驾驶事故；
- 不可靠性；
- 批评；
- 法律或监管问题；
- 公众担忧；
- 发展受阻；
- 技术局限。


重要：

请判断的是文本对自动驾驶本身的情感倾向，
而不是新闻事件整体的一般情绪。

不要根据：
- 企业股价；
- 公司整体经营情况；
- 与自动驾驶无关的商业信息

进行判断。

只能根据提供的文本进行判断。

不要使用外部知识。


只返回一个数字：

1
0
或
-1


新闻文本：

{text}

""".strip()

In [7]:
# %%
# ============================================================
# 6. Create Batch input
# ============================================================


with open(
    BATCH_INPUT_PATH,
    "w",
    encoding="utf-8"
) as f:


    for idx,row in sentiment_df.iterrows():


        request = {


            "custom_id":
                f"xinhua_{idx}",


            "method":
                "POST",


            "url":
                "/v1/responses",


            "body":{


                "model":
                    MODEL,


                "input":
                    build_prompt_p2_cn(
                        row["text"]
                    )

            }

        }


        f.write(
            json.dumps(
                request,
                ensure_ascii=False
            )
            +
            "\n"
        )


print(
    "Batch input created:"
)

print(
    BATCH_INPUT_PATH
)

Batch input created:
/Users/yurujia/Desktop/Dissertation Data/China/xinhua_sentiment_P2_CN_batch_input.jsonl


In [8]:
# %%
# ============================================================
# 7. Upload
# ============================================================


batch_file = client.files.create(

    file=open(
        BATCH_INPUT_PATH,
        "rb"
    ),

    purpose="batch"

)


print(
    batch_file.id
)

file-MR5gUnwDhxavCww7ZeExew


In [9]:
# %%
# ============================================================
# 8. Create Batch Job
# ============================================================


batch_job = client.batches.create(

    input_file_id=
        batch_file.id,


    endpoint=
        "/v1/responses",


    completion_window=
        "24h"

)


print(
    "Batch ID:"
)

print(
    batch_job.id
)

Batch ID:
batch_6a64d5738f708190af331f1ed3ed8657


In [14]:
# %%
status = client.batches.retrieve(
    batch_job.id
)


print(
    status.status
)

completed


In [15]:
# %%
# ============================================================
# 10. Download batch output
# ============================================================


status = client.batches.retrieve(
    batch_job.id
)


output_file_id = (
    status.output_file_id
)



file_response = client.files.content(
    output_file_id
)



with open(
    BATCH_OUTPUT_PATH,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        file_response.text
    )


print(
    "Saved:"
)

print(
    BATCH_OUTPUT_PATH
)

Saved:
/Users/yurujia/Desktop/Dissertation Data/China/xinhua_sentiment_P2_CN_batch_output.jsonl


In [16]:
# %%
# ============================================================
# 11. Parse results
# ============================================================


predictions = []


with open(
    BATCH_OUTPUT_PATH,
    encoding="utf-8"
) as f:


    for line in f:


        item = json.loads(line)



        article_id = int(
            item["custom_id"]
            .replace(
                "xinhua_",
                ""
            )
        )



        body = (
            item["response"]
            ["body"]
        )



        output_text = None



        for output_item in body["output"]:


            if (
                output_item.get("type")
                ==
                "message"
            ):


                for content_item in (
                    output_item.get(
                        "content",
                        []
                    )
                ):


                    if (
                        content_item.get("type")
                        ==
                        "output_text"
                    ):

                        output_text = (
                            content_item["text"]
                        )



        if output_text is None:

            print(
                "No output:",
                article_id
            )

            continue



        output_text = (
            output_text.strip()
        )



        if output_text in [
            "1",
            "0",
            "-1"
        ]:


            predictions.append({

                "article_id":
                    article_id,


                "sentiment_P2_CN":
                    int(output_text)

            })


        else:

            print(
                "Unexpected:",
                article_id,
                output_text
            )



prediction_df = pd.DataFrame(
    predictions
)


print(
    "Parsed:",
    len(prediction_df)
)


display(
    prediction_df.head()
)

Parsed: 1155


,article_id,sentiment_P2_CN
0,0,1
1,1,1
2,2,1
3,3,1
4,4,1


In [17]:
# %%
# ============================================================
# 12. Merge
# ============================================================


final_results = (
    sentiment_df
    .merge(
        prediction_df,
        on="article_id",
        how="left"
    )
)



final_results["model"] = MODEL


final_results["prompt"] = (
    "P2-CN"
)


final_results["classification_date"] = (
    pd.Timestamp.now()
)



print(
    "Missing:",
    final_results[
        "sentiment_P2_CN"
    ]
    .isna()
    .sum()
)


display(
    final_results.head()
)

Missing: 0


,article_id,source,text,sentiment_P2_CN,model,prompt,classification_date
0,0,Xinhua,在李克强总理访问德国期间，蔚来汽车５月３１日在柏林与德国大陆集团签署战略合作协议，主要涉及电...,1,gpt-5-mini,P2-CN,2026-07-25 23:30:40.879778
1,1,Xinhua,王卫东说，中德双方应“共塑创新”，构建双边关系发展新引擎。双方在智能制造、人工智能、自动驾驶...,1,gpt-5-mini,P2-CN,2026-07-25 23:30:40.879778
2,2,Xinhua,“我们正在逐渐把自动驾驶融入电动汽车这个重大专项、融入未来电动汽车产品当中。”万钢说。,1,gpt-5-mini,P2-CN,2026-07-25 23:30:40.879778
3,3,Xinhua,吉利欧洲研发中心首席执行官方浩瀚日前在位于瑞典哥德堡的总部对新华社记者表示，未来汽车应用自动...,1,gpt-5-mini,P2-CN,2026-07-25 23:30:40.879778
4,4,Xinhua,下一代超高速无线通信技术——第五代移动通信技术（５Ｇ）的商业使用区域预计将于２０２３年扩大至...,1,gpt-5-mini,P2-CN,2026-07-25 23:30:40.879778


In [18]:
# %%
# ============================================================
# 13. Attach to original dataset
# ============================================================


df["article_id"] = range(
    len(df)
)



df_final = (
    df
    .merge(
        final_results[
            [
                "article_id",
                "sentiment_P2_CN",
                "model",
                "prompt",
                "classification_date"
            ]
        ],
        on="article_id",
        how="left"
    )
)



display(
    df_final.head()
)

,file,title,source,date,content,source_file,has_body_marker,has_wan_marker,has_load_date_marker,article_content_clean,...,first_av_relevant_paragraph_english_word_count,first_av_relevant_paragraph_number_count,first_av_relevant_paragraph_approx_text_unit_count,first_paragraph_is_av_relevant,av_paragraph_same_as_original_first,article_id,sentiment_P2_CN,model,prompt,classification_date
0,（财经）专访：“中德电动汽车合作远大于竞争”——访蔚来汽车董事长李斌.DOCX,（财经）专访：“中德电动汽车合作远大于竞争”——访蔚来汽车董事长李斌,Xinhua: News in Chinese for Overseas Service,"June 2, 2017 Friday 3:43 AM GMT",（财经）专访：“中德电动汽车合作远大于竞争”——访蔚来汽车董事长李斌\nXinhua: Ne...,lexis_structured1-200,True,True,True,新华社柏林６月１日电专访：“中德电动汽车合作远大于竞争”——访蔚来汽车董事长李斌\n新华社记...,...,0,3,117,False,False,0,1,gpt-5-mini,P2-CN,2026-07-25 23:30:40.879778
1,（李克强出访配合稿）专访：务实合作始终是中德关系的基本特征——访中国驻德国公使衔参赞王卫东....,（李克强出访配合稿）专访：务实合作始终是中德关系的基本特征——访中国驻德国公使衔参赞王卫东,Xinhua: News in Chinese for Overseas Service,"June 3, 2017 Saturday 4:15 AM GMT",（李克强出访配合稿）专访：务实合作始终是中德关系的基本特征——访中国驻德国公使衔参赞王卫东\...,lexis_structured1-200,True,True,True,新华社柏林６月２日电专访：务实合作始终是中德关系的基本特征——访中国驻德国公使衔参赞王卫东\...,...,0,0,122,False,False,1,1,gpt-5-mini,P2-CN,2026-07-25 23:30:40.879778
2,中国新能源汽车全球占比一半.DOCX,中国新能源汽车全球占比一半,Xinhua: News in Chinese for Overseas Service,"June 6, 2017 Tuesday 8:29 AM GMT",中国新能源汽车全球占比一半\nXinhua: News in Chinese for Ove...,lexis_structured1-200,True,True,True,新华社北京６月６日电（记者陈芳 董瑞丰）通过手机定位，找到距离最近的电动汽车，用手机开锁后就...,...,0,0,38,False,False,2,1,gpt-5-mini,P2-CN,2026-07-25 23:30:40.879778
3,（科技）专家：自动驾驶技术有望让交通事故零伤亡.DOCX,（科技）专家：自动驾驶技术有望让交通事故零伤亡,Xinhua: News in Chinese for Overseas Service,"June 7, 2017 Wednesday 7:24 AM GMT",（科技）专家：自动驾驶技术有望让交通事故零伤亡\nXinhua: News in Chine...,lexis_structured1-200,True,True,True,新华社瑞典哥德堡６月７日电（记者潘革平 付一鸣）吉利欧洲研发中心首席执行官方浩瀚日前在位于瑞...,...,0,1,73,True,True,3,1,gpt-5-mini,P2-CN,2026-07-25 23:30:40.879778
4,（财经）日本预计于２０２３年普及５Ｇ通信.DOCX,（财经）日本预计于２０２３年普及５Ｇ通信,Xinhua: News in Chinese for Overseas Service,"June 7, 2017 Wednesday 6:42 AM GMT",（财经）日本预计于２０２３年普及５Ｇ通信\nXinhua: News in Chinese ...,lexis_structured1-200,True,True,True,新华社东京６月７日电（记者钱铮）下一代超高速无线通信技术——第五代移动通信技术（５Ｇ）的商业...,...,0,2,62,True,True,4,1,gpt-5-mini,P2-CN,2026-07-25 23:30:40.879778


In [19]:
# %%
# ============================================================
# 14. Save
# ============================================================


df_final.to_excel(
    OUTPUT_PATH,
    index=False
)


print(
    "Final saved:"
)

print(
    OUTPUT_PATH
)

Final saved:
/Users/yurujia/Desktop/Dissertation Data/China/xinhua_sentiment_P2_CN_final.xlsx
